In [1]:
# 高性能VAE缺失值填补模型 - H和V特征专用版本
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers, optimizers, regularizers, initializers, callbacks
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, QuantileTransformer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy import stats
from scipy import special
import warnings
import random
import os
import gc
import time

warnings.filterwarnings("ignore")

# 检查TensorFlow版本
print(f"TensorFlow版本: {tf.__version__}")

# ==================== 强化的可重复性设置 ====================
SEED = 42

def set_all_seeds(seed=42):
    """设置所有随机种子以确保可重复性"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# 初始化随机种子
set_all_seeds(SEED)

# GPU内存设置
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"发现 {len(gpus)} 个GPU设备")
    except RuntimeError as e:
        print(f"GPU设置错误: {e}")
else:
    print("未发现GPU，使用CPU运行")

# 清理会话
tf.keras.backend.clear_session()
gc.collect()

# 手动实现反Box-Cox变换
def inv_boxcox(y, lambda_param):
    """手动实现反Box-Cox变换"""
    if lambda_param == 0:
        return np.exp(y)
    else:
        # 避免负值导致的问题
        return np.power(lambda_param * y + 1, 1 / lambda_param)

# 1. 数据加载和预处理
def load_and_preprocess_data():
    """加载数据并进行初步预处理"""
    data = pd.read_csv("./Others.csv", low_memory=True)
    print(f"原始数据形状: {data.shape}")
    
    # 保存原始数据副本
    original_data = data.copy()
    
    # 删除非数值列
    cols_to_drop = []
    for col in data.columns:
        try:
            pd.to_numeric(data[col].dropna().iloc[:10])
        except:
            cols_to_drop.append(col)
    
    data = data.drop(columns=cols_to_drop)
    
    # 删除ID列
    for col in data.columns[:3]:
        if data[col].dtype in ['int64', 'float64']:
            non_null = data[col].dropna()
            if len(non_null) > 1:
                diffs = non_null.diff().dropna()
                if (diffs == 1).mean() > 0.9:
                    data = data.drop(columns=[col])
    
    # 删除L(m)列
    if 'L(m)' in data.columns:
        data = data.drop(columns=['L(m)'])
    
    # 移除缺失率过高的列（>30%）
    missing_ratio = data.isnull().sum() / len(data)
    cols_to_drop = missing_ratio[missing_ratio > 0.30].index
    data = data.drop(columns=cols_to_drop)
    
    # 转换数据类型
    for col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce').astype('float32')
    
    print(f"最终数据形状: {data.shape}")
    print(f"特征: {list(data.columns)}")
    
    # 验证是否只有H和V两列
    if len(data.columns) == 2:
        print(f"确认数据包含 {len(data.columns)} 个特征")
    
    return data, original_data

# 2. 针对H和V的异常值处理
def enhanced_outlier_handling(df):
    """针对H和V特征的专门异常值处理"""
    df_clean = df.copy()
    transform_info = {}
    
    for column in df_clean.columns:
        mask = df_clean[column].notna()
        values = df_clean.loc[mask, column].values
        
        if len(values) == 0:
            continue
        
        # H特征处理：Box-Cox + IQR
        if 'H' in column:
            print(f"{column}: 使用Box-Cox + IQR处理")
            
            # 确保数据为正
            min_val = values.min()
            if min_val <= 0:
                shift = 1 - min_val
                values = values + shift
            else:
                shift = 0
            
            # Box-Cox变换
            values_transformed, lambda_param = stats.boxcox(values)
            
            # IQR异常值处理
            q1, q3 = np.percentile(values_transformed, [25, 75])
            iqr = q3 - q1
            lower = q1 - 3 * iqr  # 使用3倍IQR（温和处理）
            upper = q3 + 3 * iqr
            values_transformed = np.clip(values_transformed, lower, upper)
            
            transform_info[column] = {'type': 'boxcox', 'lambda': lambda_param, 'shift': shift}
            df_clean.loc[mask, column] = values_transformed
            print(f"  Box-Cox lambda={lambda_param:.4f}, shift={shift:.4f}")
            print(f"  IQR裁剪范围: [{lower:.2f}, {upper:.2f}]")
            
        # V特征处理：Quantile变换
        elif 'V' in column:
            print(f"{column}: 使用Quantile变换处理极端分布")
            
            # 使用Quantile变换处理极端偏斜分布
            qt = QuantileTransformer(
                n_quantiles=min(1000, len(values)), 
                output_distribution='normal', 
                random_state=SEED
            )
            values_transformed = qt.fit_transform(values.reshape(-1, 1)).flatten()
            
            transform_info[column] = {'type': 'quantile', 'transformer': qt}
            df_clean.loc[mask, column] = values_transformed
            print(f"  Quantile变换完成，输出为标准正态分布")
            
        else:
            # 其他特征（如果有）：默认IQR处理
            print(f"{column}: 使用标准IQR异常值处理")
            
            q1, q3 = np.percentile(values, [25, 75])
            iqr = q3 - q1
            lower = q1 - 2.5 * iqr
            upper = q3 + 2.5 * iqr
            values = np.clip(values, lower, upper)
            
            transform_info[column] = {'type': 'clip', 'shift': 0}
            df_clean.loc[mask, column] = values
    
    return df_clean, transform_info

# 3. 针对H和V的标准化
def enhanced_normalization(data):
    """针对H和V特征的标准化方法"""
    data_normalized = pd.DataFrame(index=data.index, columns=data.columns)
    scalers = {}
    
    for column in data.columns:
        mask = data[column].notna()
        values = data[column][mask].values.reshape(-1, 1)
        
        if len(values) > 0:
            # 根据特征选择不同的标准化方法
            if 'V' in column:
                # V使用StandardScaler（因为已经做了Quantile变换）
                scaler = StandardScaler()
                print(f"{column}: 使用StandardScaler")
            elif 'H' in column:
                # H使用RobustScaler（因为做了Box-Cox变换）
                scaler = RobustScaler(quantile_range=(10, 90))
                print(f"{column}: 使用RobustScaler")
            else:
                # 默认使用RobustScaler
                scaler = RobustScaler(quantile_range=(5, 95))
            
            scaler.fit(values)
            scaled = scaler.transform(values).flatten()
            
            # 映射到[0.1, 0.9]以避免边界效应
            min_val, max_val = scaled.min(), scaled.max()
            if max_val - min_val > 1e-8:
                scaled = (scaled - min_val) / (max_val - min_val) * 0.8 + 0.1
            else:
                scaled = np.ones_like(scaled) * 0.5
            
            data_normalized.loc[mask, column] = scaled.astype(np.float32)
            scalers[column] = {'scaler': scaler, 'min': min_val, 'max': max_val}
    
    return data_normalized, scalers

# 修复的反标准化函数
def inverse_transform_array(data_normalized_array, scalers, transform_info, original_columns, original_index=None):
    """将标准化的numpy数组转换回原始尺度的DataFrame"""
    n_samples = data_normalized_array.shape[0]
    
    if original_index is None:
        original_index = range(n_samples)
    
    data_original = pd.DataFrame(index=original_index, columns=original_columns)
    
    for i, column in enumerate(original_columns):
        if column not in scalers:
            data_original[column] = data_normalized_array[:, i]
            continue
            
        values = data_normalized_array[:, i].copy()
        
        # 反标准化
        scaler_info = scalers[column]
        min_val = scaler_info['min']
        max_val = scaler_info['max']
        
        # 从[0.1, 0.9]映射回原始范围
        values = (values - 0.1) / 0.8 * (max_val - min_val) + min_val
        
        # 反Scaler
        values = scaler_info['scaler'].inverse_transform(values.reshape(-1, 1)).flatten()
        
        # 反变换
        if column in transform_info:
            trans_type = transform_info[column]['type']
            
            if trans_type == 'quantile':
                # 反Quantile变换
                transformer = transform_info[column]['transformer']
                values = transformer.inverse_transform(values.reshape(-1, 1)).flatten()
            elif trans_type == 'boxcox':
                # 反Box-Cox变换（使用手动实现的函数）
                lambda_param = transform_info[column]['lambda']
                shift = transform_info[column]['shift']
                values = inv_boxcox(values, lambda_param) - shift
        
        data_original[column] = values
    
    return data_original

# 4. 增强的VAE模型（适配2个特征）
class EnhancedVAE:
    def __init__(self, data_dim, latent_dim=4, learning_rate=0.0005, seed=42):
        """增强的VAE模型，针对2个特征优化"""
        self.data_dim = data_dim
        self.latent_dim = latent_dim  # 对于2个特征，减小潜在维度
        self.learning_rate = learning_rate
        self.seed = seed
        
        tf.random.set_seed(seed)
        
        # 构建增强的模型
        self.encoder = self._build_encoder()
        self.decoder = self._build_decoder()
        
        # 使用Adam优化器
        self.optimizer = optimizers.Adam(learning_rate=learning_rate)
        
        self.best_weights = None
        self.best_val_loss = float('inf')
    
    def _build_encoder(self):
        """构建编码器（针对2个特征优化）"""
        inputs = layers.Input(shape=(self.data_dim,))
        
        # 适配2个特征的网络结构
        x = layers.Dense(32, kernel_initializer='he_normal')(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.2)(x)
        
        x = layers.Dense(16, kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.15)(x)
        
        x = layers.Dense(8, kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        
        z_mean = layers.Dense(self.latent_dim)(x)
        z_log_var = layers.Dense(self.latent_dim)(x)
        
        # 限制log_var的范围
        z_log_var = layers.Lambda(lambda x: tf.clip_by_value(x, -3, 3))(z_log_var)
        
        return keras.Model(inputs, [z_mean, z_log_var])
    
    def _build_decoder(self):
        """构建解码器（针对2个特征优化）"""
        inputs = layers.Input(shape=(self.latent_dim,))
        
        x = layers.Dense(8, kernel_initializer='he_normal')(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        
        x = layers.Dense(16, kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.15)(x)
        
        x = layers.Dense(32, kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.2)(x)
        
        outputs = layers.Dense(self.data_dim, activation='sigmoid')(x)
        
        return keras.Model(inputs, outputs)
    
    def _sample(self, z_mean, z_log_var):
        """重参数化技巧"""
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon * 0.05  # 减小噪声
    
    @tf.function
    def train_step(self, x_batch, mask_batch, kl_weight=0.0005):
        """训练步骤"""
        with tf.GradientTape() as tape:
            # 编码
            z_mean, z_log_var = self.encoder(x_batch, training=True)
            z = self._sample(z_mean, z_log_var)
            
            # 解码
            reconstructed = self.decoder(z, training=True)
            
            # 重建损失（只计算观测值）
            mse = tf.square(x_batch - reconstructed) * mask_batch
            reconstruction_loss = tf.reduce_sum(mse) / tf.reduce_sum(mask_batch)
            
            # KL损失
            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
            )
            
            # 总损失
            total_loss = reconstruction_loss + kl_weight * kl_loss
        
        # 梯度更新
        variables = self.encoder.trainable_variables + self.decoder.trainable_variables
        gradients = tape.gradient(total_loss, variables)
        gradients = [tf.clip_by_norm(g, 1.0) for g in gradients]
        self.optimizer.apply_gradients(zip(gradients, variables))
        
        return total_loss
    
    def fit(self, data, mask, validation_split=0.2, epochs=500, batch_size=64):
        """训练模型"""
        # 数据分割
        n_samples = len(data)
        indices = np.arange(n_samples)
        np.random.seed(self.seed)
        np.random.shuffle(indices)
        
        split_point = int(n_samples * (1 - validation_split))
        train_indices = indices[:split_point]
        val_indices = indices[split_point:]
        
        x_train = data[train_indices]
        x_val = data[val_indices]
        mask_train = mask[train_indices]
        mask_val = mask[val_indices]
        
        print(f"训练集: {len(x_train)} 样本")
        print(f"验证集: {len(x_val)} 样本")
        
        # 早停设置
        patience = 50
        patience_counter = 0
        
        # 学习率调度
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            self.learning_rate,
            decay_steps=1000,
            decay_rate=0.96,
            staircase=True
        )
        self.optimizer.learning_rate = lr_schedule
        
        for epoch in range(epochs):
            # 动态调整KL权重
            kl_weight = min(0.001, 0.0001 * (epoch / 100))
            
            # 训练
            train_indices_shuffled = np.random.permutation(len(x_train))
            train_loss = 0
            n_batches = 0
            
            for i in range(0, len(x_train), batch_size):
                batch_indices = train_indices_shuffled[i:i+batch_size]
                loss = self.train_step(
                    x_train[batch_indices],
                    mask_train[batch_indices],
                    kl_weight
                )
                train_loss += loss
                n_batches += 1
            
            train_loss /= n_batches
            
            # 验证
            if epoch % 10 == 0:
                val_loss = 0
                n_val_batches = 0
                
                for i in range(0, len(x_val), batch_size):
                    z_mean, z_log_var = self.encoder(x_val[i:i+batch_size], training=False)
                    z = self._sample(z_mean, z_log_var)
                    reconstructed = self.decoder(z, training=False)
                    
                    mse = tf.square(x_val[i:i+batch_size] - reconstructed) * mask_val[i:i+batch_size]
                    val_loss += tf.reduce_sum(mse) / tf.reduce_sum(mask_val[i:i+batch_size])
                    n_val_batches += 1
                
                val_loss /= n_val_batches
                
                if val_loss < self.best_val_loss:
                    self.best_val_loss = val_loss
                    self.best_weights = {
                        'encoder': self.encoder.get_weights(),
                        'decoder': self.decoder.get_weights()
                    }
                    patience_counter = 0
                else:
                    patience_counter += 10
                
                if epoch % 50 == 0:
                    print(f"Epoch {epoch}: Train Loss={train_loss:.6f}, Val Loss={val_loss:.6f}")
                
                # 早停
                if patience_counter >= patience:
                    print(f"早停于第 {epoch} 轮")
                    break
        
        # 恢复最佳权重
        if self.best_weights:
            self.encoder.set_weights(self.best_weights['encoder'])
            self.decoder.set_weights(self.best_weights['decoder'])
    
    def impute(self, data, mask):
        """填补缺失值"""
        imputed = np.zeros_like(data)
        batch_size = 128
        
        for i in range(0, len(data), batch_size):
            batch_data = data[i:i+batch_size]
            batch_mask = mask[i:i+batch_size]
            
            # 多次采样取平均
            predictions = []
            for _ in range(20):  # 增加采样次数
                z_mean, z_log_var = self.encoder(batch_data, training=False)
                z = self._sample(z_mean, z_log_var)
                pred = self.decoder(z, training=False).numpy()
                predictions.append(pred)
            
            pred_mean = np.mean(predictions, axis=0)
            
            # 混合原始值和预测值
            imputed[i:i+batch_size] = batch_data * batch_mask + pred_mean * (1 - batch_mask)
        
        return imputed

# 5. 特征专属的深度学习模型（简化版）
class FeatureSpecificDeepModels:
    def __init__(self, seed=42):
        self.seed = seed
        self.models = {}
        self.scalers = {}
    
    def _build_feature_model(self, input_dim):
        """为单个特征构建深度学习模型（简化版）"""
        model = keras.Sequential([
            layers.Dense(16, activation='relu', input_shape=(input_dim,)),
            layers.Dropout(0.2),
            layers.Dense(8, activation='relu'),
            layers.Dropout(0.1),
            layers.Dense(4, activation='relu'),
            layers.Dense(1)
        ])
        model.compile(optimizer='adam', loss='mse')
        return model
    
    def fit(self, data, mask, column_names=None):
        """为每个特征训练专门的深度模型"""
        print("训练特征专属深度模型...")
        
        for col_idx in range(data.shape[1]):
            col_name = column_names[col_idx] if column_names is not None else f"feature_{col_idx}"
            obs_mask = mask[:, col_idx] == 1
            
            if np.sum(obs_mask) < 100:
                continue
            
            # 准备训练数据（只有另一个特征作为输入）
            X_train = np.delete(data[obs_mask], col_idx, axis=1)
            y_train = data[obs_mask, col_idx]
            
            # 标准化
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
            X_train_scaled = scaler_x.fit_transform(X_train.reshape(-1, 1))  # 只有1个输入特征
            y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
            
            # 构建和训练模型
            model = self._build_feature_model(1)  # 输入维度为1
            
            # 早停回调
            early_stop = callbacks.EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
            
            model.fit(
                X_train_scaled, y_train_scaled,
                epochs=100,
                batch_size=32,
                verbose=0,
                callbacks=[early_stop]
            )
            
            self.models[col_idx] = model
            self.scalers[col_idx] = {'x': scaler_x, 'y': scaler_y}
            print(f"  完成 {col_name} 的模型训练")
    
    def impute(self, data, mask):
        """填补缺失值"""
        imputed = data.copy()
        
        for col_idx, model in self.models.items():
            miss_mask = mask[:, col_idx] == 0
            if np.sum(miss_mask) == 0:
                continue
            
            X_miss = np.delete(imputed[miss_mask], col_idx, axis=1)
            
            # 标准化
            X_miss_scaled = self.scalers[col_idx]['x'].transform(X_miss.reshape(-1, 1))
            
            # 预测
            pred_scaled = model.predict(X_miss_scaled, verbose=0).flatten()
            
            # 反标准化
            pred = self.scalers[col_idx]['y'].inverse_transform(pred_scaled.reshape(-1, 1)).flatten()
            
            imputed[miss_mask, col_idx] = pred
        
        return imputed

# 6. 针对H和V的集成模型
class EnhancedEnsemble:
    def __init__(self, data_dim, seed=42):
        self.data_dim = data_dim
        self.seed = seed
        self.models = []
        self.weights = []
        self.feature_weights = {}  # 特征专属权重
    
    def fit(self, data, mask, column_names):
        """训练集成模型（针对H和V优化）"""
        print("\n训练增强集成模型...")
        
        # 1. KNN（快速基线）
        print("1. KNN模型...")
        knn = KNNImputer(n_neighbors=15, weights='distance')
        knn.fit(data)
        self.models.append(('knn', knn))
        
        # 2. 高级IterativeImputer - GradientBoosting
        print("2. GradientBoosting IterativeImputer...")
        gb_mice = IterativeImputer(
            estimator=GradientBoostingRegressor(
                n_estimators=150,
                max_depth=5,
                learning_rate=0.1,
                subsample=0.8,
                random_state=self.seed
            ),
            max_iter=25,
            random_state=self.seed
        )
        gb_mice.fit(data)
        self.models.append(('gb_mice', gb_mice))
        
        # 3. 随机森林IterativeImputer
        print("3. 随机森林IterativeImputer...")
        rf_mice = IterativeImputer(
            estimator=RandomForestRegressor(
                n_estimators=150,
                max_depth=8,
                min_samples_split=3,
                min_samples_leaf=2,
                random_state=self.seed
            ),
            max_iter=20,
            random_state=self.seed
        )
        rf_mice.fit(data)
        self.models.append(('rf_mice', rf_mice))
        
        # 4. ExtraTrees IterativeImputer
        print("4. ExtraTrees IterativeImputer...")
        et_mice = IterativeImputer(
            estimator=ExtraTreesRegressor(
                n_estimators=150,
                max_depth=8,
                min_samples_split=3,
                random_state=self.seed
            ),
            max_iter=20,
            random_state=self.seed
        )
        et_mice.fit(data)
        self.models.append(('et_mice', et_mice))
        
        # 5. 特征专属深度模型
        print("5. 特征专属深度模型...")
        fs_model = FeatureSpecificDeepModels(seed=self.seed)
        fs_model.fit(data, mask, column_names)
        self.models.append(('feature_deep', fs_model))
        
        # 6. 增强VAE
        print("6. 增强VAE模型...")
        vae = EnhancedVAE(data_dim=self.data_dim, latent_dim=4, learning_rate=0.0005, seed=self.seed)
        
        # 初始填补
        mean_imputer = SimpleImputer(strategy='mean')
        mean_imputer.fit(data)
        initial_imputed = mean_imputer.transform(data)
        data_filled = np.where(mask == 1, data, initial_imputed)
        
        vae.fit(data_filled, mask, epochs=500, batch_size=64)
        self.models.append(('vae', vae))
        
        # 设置特征专属权重（针对H和V）
        for i, col_name in enumerate(column_names):
            if 'H' in col_name:
                # H特征的权重
                self.feature_weights[i] = {
                    'knn': 0.15,
                    'gb_mice': 0.20,
                    'rf_mice': 0.20,
                    'et_mice': 0.15,
                    'feature_deep': 0.20,
                    'vae': 0.10
                }
            elif 'V' in col_name:
                # V特征的权重
                self.feature_weights[i] = {
                    'knn': 0.15,
                    'gb_mice': 0.25,
                    'rf_mice': 0.25,
                    'et_mice': 0.20,
                    'feature_deep': 0.10,
                    'vae': 0.05
                }
            else:
                # 默认权重（如果有其他特征）
                self.feature_weights[i] = {
                    'knn': 0.17,
                    'gb_mice': 0.17,
                    'rf_mice': 0.17,
                    'et_mice': 0.17,
                    'feature_deep': 0.16,
                    'vae': 0.16
                }
        
        print("集成模型训练完成")
    
    def impute(self, data, mask):
        """增强的集成预测"""
        n_samples, n_features = data.shape
        imputed = np.zeros_like(data)
        
        # 获取所有模型的预测
        all_predictions = {}
        for name, model in self.models:
            if name == 'vae':
                mean_imputer = SimpleImputer(strategy='mean')
                mean_imputer.fit(data)
                initial_imputed = mean_imputer.transform(data)
                data_filled = np.where(mask == 1, data, initial_imputed)
                pred = model.impute(data_filled, mask)
            elif name == 'feature_deep':
                pred = model.impute(data, mask)
            else:
                pred = model.transform(data)
            all_predictions[name] = pred
        
        # 对每个特征使用专属权重进行加权平均
        for feature_idx in range(n_features):
            if feature_idx in self.feature_weights:
                weights = self.feature_weights[feature_idx]
                feature_pred = np.zeros(n_samples)
                
                for model_name, weight in weights.items():
                    if model_name in all_predictions:
                        feature_pred += all_predictions[model_name][:, feature_idx] * weight
                
                imputed[:, feature_idx] = feature_pred
            else:
                # 如果没有专属权重，使用均值
                feature_pred = np.zeros(n_samples)
                n_models = len(all_predictions)
                for pred in all_predictions.values():
                    feature_pred += pred[:, feature_idx] / n_models
                imputed[:, feature_idx] = feature_pred
        
        # 确保观测值不变
        imputed = data * mask + imputed * (1 - mask)
        
        return imputed

# 7. 评估函数
def evaluate_imputation_original_scale(model, data_array, mask, column_names, 
                                      data_raw, scalers, transform_info):
    """使用原始尺度的交叉验证评估模型"""
    print("\n=== 使用原始尺度评估模型性能 ===")
    
    results = []
    all_observations = []
    
    original_index = data_raw.index
    
    for col_idx, col_name in enumerate(column_names):
        obs_mask = mask[:, col_idx] == 1
        obs_indices = np.where(obs_mask)[0]
        
        if len(obs_indices) < 100:
            continue
        
        # 使用5折交叉验证
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        
        all_true = []
        all_pred = []
        all_indices = []
        
        for train_idx, test_idx in kf.split(obs_indices):
            test_indices = obs_indices[test_idx]
            
            # 创建测试掩码
            test_mask = mask.copy()
            test_mask[test_indices, col_idx] = 0
            
            # 填补（标准化尺度）
            imputed_normalized = model.impute(data_array, test_mask)
            
            # 转换回原始尺度
            imputed_original = inverse_transform_array(
                imputed_normalized, scalers, transform_info, 
                column_names, original_index
            )
            
            # 获取原始尺度的真实值
            true_values_original = data_raw.iloc[test_indices][col_name].values
            pred_values_original = imputed_original[col_name].iloc[test_indices].values
            
            all_true.extend(true_values_original)
            all_pred.extend(pred_values_original)
            all_indices.extend(test_indices)
            
            # 记录每个预测
            for idx, true_val, pred_val in zip(test_indices, true_values_original, pred_values_original):
                all_observations.append({
                    '行索引': idx,
                    '特征': col_name,
                    '原始观测值': true_val,
                    '预测值': pred_val,
                    '绝对误差': abs(true_val - pred_val)
                })
        
        all_true = np.array(all_true)
        all_pred = np.array(all_pred)
        
        # 使用原始尺度计算指标
        mae = mean_absolute_error(all_true, all_pred)
        rmse = np.sqrt(mean_squared_error(all_true, all_pred))
        
        # 标准R²计算
        ss_res = np.sum((all_true - all_pred) ** 2)
        ss_tot = np.sum((all_true - np.mean(all_true)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        # 相关系数
        if np.std(all_true) > 0 and np.std(all_pred) > 0:
            correlation = np.corrcoef(all_true, all_pred)[0, 1]
        else:
            correlation = 0
        
        results.append({
            '特征': col_name,
            'R²(原始尺度)': r2,
            '相关系数': correlation,
            'MAE': mae,
            'RMSE': rmse,
            '样本数': len(all_true)
        })
        
        print(f"{col_name}: R²={r2:.4f}, 相关系数={correlation:.4f}, MAE={mae:.4f}")
    
    return pd.DataFrame(results), pd.DataFrame(all_observations)

# 8. 主函数
def main():
    """主函数"""
    print("=" * 80)
    print("VAE缺失值填补模型 - H和V特征专用版本")
    print("H特征：Box-Cox + IQR处理")
    print("V特征：Quantile变换处理")
    print("=" * 80)
    
    try:
        set_all_seeds(SEED)
        
        # 1. 加载数据
        print("\n1. 加载数据...")
        data_raw, original_data = load_and_preprocess_data()
        
        # 保存原始数据用于评估
        data_raw_copy = data_raw.copy()
        
        # 2. 异常值处理
        print("\n2. 异常值处理...")
        data_clean, transform_info = enhanced_outlier_handling(data_raw)
        
        # 3. 数据标准化
        print("\n3. 数据标准化...")
        data_normalized, scalers = enhanced_normalization(data_clean)
        
        # 创建掩码
        mask = (~data_normalized.isnull()).astype(np.float32).values
        data_array = data_normalized.fillna(0.5).values.astype(np.float32)
        
        print(f"\n数据形状: {data_array.shape}")
        print(f"缺失率: {(1 - mask.mean())*100:.2f}%")
        
        # 4. 训练集成模型
        print("\n4. 训练集成模型...")
        model = EnhancedEnsemble(data_dim=data_array.shape[1], seed=SEED)
        model.fit(data_array, mask, data_normalized.columns)
        
        # 5. 使用原始尺度评估
        print("\n5. 使用原始尺度评估模型...")
        eval_results, observations_df = evaluate_imputation_original_scale(
            model, data_array, mask, data_normalized.columns, 
            data_raw_copy, scalers, transform_info
        )
        
        # 6. 最终填补
        print("\n6. 生成最终填补数据...")
        final_imputed_normalized = model.impute(data_array, mask)
        
        # 转换回原始尺度
        final_imputed_original = inverse_transform_array(
            final_imputed_normalized, scalers, transform_info, 
            data_normalized.columns, data_raw_copy.index
        )
        
        # 创建完整的填补结果
        final_result = data_raw_copy.copy()
        for col in final_result.columns:
            missing_mask = data_raw_copy[col].isna()
            final_result.loc[missing_mask, col] = final_imputed_original.loc[missing_mask, col]
        
        # 7. 输出结果
        print("\n" + "=" * 80)
        print("模型性能总结（原始尺度）")
        print("=" * 80)
        
        for _, row in eval_results.iterrows():
            print(f"\n{row['特征']}:")
            print(f"  R²(原始尺度): {row['R²(原始尺度)']:.4f}")
            print(f"  相关系数: {row['相关系数']:.4f}")
            print(f"  MAE: {row['MAE']:.4f}")
            print(f"  RMSE: {row['RMSE']:.4f}")
            print(f"  评估样本数: {row['样本数']}")
        
        print(f"\n平均R²(原始尺度): {eval_results['R²(原始尺度)'].mean():.4f}")
        print(f"最低R²(原始尺度): {eval_results['R²(原始尺度)'].min():.4f}")
        
        # 保存结果
        observations_df.to_csv("观测值与预测值_HV版本.csv", index=False, encoding='utf-8-sig')
        final_result.to_csv("完整填补结果_HV版本.csv", index=False, encoding='utf-8-sig')
        eval_results.to_csv("评估结果汇总_HV版本.csv", index=False, encoding='utf-8-sig')
        
        print("\n文件已保存：")
        print("1. 观测值与预测值_HV版本.csv")
        print("2. 完整填补结果_HV版本.csv")
        print("3. 评估结果汇总_HV版本.csv")
        
        # 检查效果
        print("\n" + "=" * 80)
        print("模型效果评估")
        print("=" * 80)
        
        target_r2 = 0.80  # 目标R²值
        
        for _, row in eval_results.iterrows():
            feature = row['特征']
            r2 = row['R²(原始尺度)']
            if r2 >= target_r2:
                print(f"✅ {feature}: R²={r2:.4f} - 达标！")
            elif r2 >= 0.70:
                print(f"⚠️ {feature}: R²={r2:.4f} - 接近达标")
            else:
                print(f"❌ {feature}: R²={r2:.4f} - 需要进一步优化")
        
        return final_result, eval_results, observations_df
        
    except Exception as e:
        print(f"\n错误: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None

if __name__ == "__main__":
    print("\n程序开始运行...")
    print("数据处理策略：")
    print("1. H特征：Box-Cox变换 + 3倍IQR异常值处理")
    print("2. V特征：Quantile变换（映射到标准正态分布）")
    print("3. VAE网络结构优化为2个特征")
    print("4. 集成6个模型进行填补")
    
    start_time = time.time()
    result, evaluation, observations = main()
    
    if result is not None:
        print(f"\n✅ 程序运行完成！")
        print(f"用时: {(time.time() - start_time)/60:.2f} 分钟")
    else:
        print("\n❌ 运行失败")
    
    gc.collect()

TensorFlow版本: 2.21.0
未发现GPU，使用CPU运行


程序开始运行...
数据处理策略：
1. H特征：Box-Cox变换 + 3倍IQR异常值处理
2. V特征：Quantile变换（映射到标准正态分布）
3. VAE网络结构优化为2个特征
4. 集成6个模型进行填补
VAE缺失值填补模型 - H和V特征专用版本
H特征：Box-Cox + IQR处理
V特征：Quantile变换处理

1. 加载数据...

错误: [Errno 2] No such file or directory: './Others.csv'

❌ 运行失败


Traceback (most recent call last):
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_5900\2079810894.py", line 849, in main
    data_raw, original_data = load_and_preprocess_data()
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_5900\2079810894.py", line 70, in load_and_preprocess_data
    data = pd.read_csv("./Others.csv", low_memory=True)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 873, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 300, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 1645, i

In [2]:
import pandas as pd

def add_L_column_to_results():
    """
    将L(m)列添加到已生成的填补结果文件中
    """
    # 1. 读取原始数据以获取L(m)列
    original_data = pd.read_csv("./Others.csv", low_memory=True)
    
    # 2. 检查L(m)列是否存在
    if 'L(m)' not in original_data.columns:
        print("原始数据中未找到L(m)列")
        return
    
    # 3. 提取L(m)列
    L_column = original_data['L(m)'].copy()
    print(f"成功提取L(m)列，共{len(L_column)}行")
    
    # 4. 读取填补结果
    try:
        imputed_results = pd.read_csv("完整填补结果_HV版本.csv")
        print(f"填补结果形状: {imputed_results.shape}")
    except FileNotFoundError:
        print("未找到填补结果文件，请先运行主程序生成结果")
        return
    
    # 5. 检查行数是否匹配
    if len(imputed_results) != len(L_column):
        print(f"警告：行数不匹配！填补结果有{len(imputed_results)}行，L(m)列有{len(L_column)}行")
        # 如果行数不同，需要对齐（取最小行数）
        min_rows = min(len(imputed_results), len(L_column))
        imputed_results = imputed_results.iloc[:min_rows]
        L_column = L_column.iloc[:min_rows]
    
    # 6. 添加L(m)列到结果中
    imputed_results['L(m)'] = L_column.values
    
    # 7. 保存更新后的结果
    output_filename = "填补结果_Others_包含L列_平衡版.csv"
    imputed_results.to_csv(output_filename, index=False)
    
    print(f"\n成功添加L(m)列到结果中！")
    print(f"新文件已保存为: {output_filename}")
    print(f"最终数据形状: {imputed_results.shape}")
    print(f"列名: {list(imputed_results.columns)}")
    
    # 8. 显示L(m)列的基本统计信息
    print("\nL(m)列统计信息:")
    print(f"  数据类型: {L_column.dtype}")
    print(f"  唯一值数量: {L_column.nunique()}")
    print(f"  缺失值数量: {L_column.isna().sum()}")
    if pd.api.types.is_numeric_dtype(L_column):
        print(f"  最小值: {L_column.min()}")
        print(f"  最大值: {L_column.max()}")
        print(f"  平均值: {L_column.mean():.2f}")
    else:
        print(f"  前5个值: {L_column.head().tolist()}")
    
    return imputed_results

# 运行函数
if __name__ == "__main__":
    result_with_L = add_L_column_to_results()

成功提取L(m)列，共579行
填补结果形状: (579, 2)

成功添加L(m)列到结果中！
新文件已保存为: 填补结果_Others_包含L列_平衡版.csv
最终数据形状: (579, 3)
列名: ['H(m)', 'V(m³)', 'L(m)']

L(m)列统计信息:
  数据类型: float64
  唯一值数量: 432
  缺失值数量: 0
  最小值: 1.17
  最大值: 1610000.0
  平均值: 7553.59
